# 02 — Modelagem e Avaliação

Treino do LightGBM, avaliação com incerteza e explicabilidade SHAP.
Reutiliza as funções de `src/` (nada de lógica de produção aqui).

In [ ]:
from warnings import filterwarnings

import numpy as np
import polars as pl
from src.config.paths import get_paths
from src.config.settings import load_settings
from src.evaluation.metrics import bootstrap_metric, evaluate_classification
from src.explainability.shap_explainer import ChurnExplainer
from src.inference.predictor import predict_proba
from src.models.persistence import load_model
from src.pipelines.common import to_pandas_xy

filterwarnings("ignore")
RANDOM_SEED = 42
rng = np.random.default_rng(seed=RANDOM_SEED)
rng.normal()

In [ ]:
settings = load_settings()
paths = get_paths()
model, meta = load_model(paths.model_file)
test = pl.read_parquet(paths.test_file)
x_test, y_test = to_pandas_xy(test)

proba = predict_proba(model, x_test)
metrics = evaluate_classification(y_test.to_numpy(), proba)
ci = bootstrap_metric(y_test.to_numpy(), proba, n_iterations=500)
print(metrics)
print(f"PR-AUC IC 95%: {ci['lower']:.3f}-{ci['upper']:.3f}")

In [ ]:
explainer = ChurnExplainer(model)
explainer.explain_global(x_test).head(12)